In [0]:
%python
# Databricks notebook source
from pyspark.sql.functions import col, to_date, row_number
from pyspark.sql.window import Window

# Step 1: Ensure silver schema exists in default catalog
spark.sql("CREATE CATALOG IF NOT EXISTS swap01")
spark.sql("CREATE SCHEMA IF NOT EXISTS swap01.bronze_schema")

# Step 2: Load Bronze orders
bronze_orders = spark.table("training_centralindia_lakeflowjobs_dbws.bronze_schema.orders_bronze")

# Step 3: Rename and cast columns
orders_cleaned = (
    bronze_orders
    .select(
        col("OrderID").alias("order_id"),
        to_date(col("OrderDate")).alias("order_date"),
        col("CustomerID").alias("customer_id"),
        col("TotalAmount").cast("double").alias("total_amount"),
        col("Status").alias("status")
    )
)

# Step 4: Write to Silver layer
orders_cleaned.write.format("delta").mode("overwrite").saveAsTable("swap01.bronze_schema.orders_cleaned")
print("✅ Silver cleaned complete: training_centralindia_lakeflowjobs_dbws.silver_schema.orders_cleaned")
